# Coastal flood step 07: mangrove attribution of avoided expected annual damages (maximum scenario)

This notebook is now a **reporting notebook** for the signed weighted area-distance attribution method.

It reads the scenario outputs created by:
- `coastal_flood_22_all_sectors_mangrove_attribution_signed_area_distance_weighting_feb_2026.ipynb`
- `coastal_flood_25_all_sectors_mangrove_attribution_signed_area_distance_weighting_maximum_scenario_feb_2026.ipynb`

Attribution method used here:
- weighted area-distance within `5000 m`
- nearest-neighbour fallback beyond the buffer
- signed attribution retained, so positive and negative avoided EADs are both visible


In [ ]:

from pathlib import Path
import sys
import pathlib

import pandas
import numpy
import geopandas
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FuncFormatter

robyn_libraries_path = pathlib.Path("../../../robyns_libraries").resolve()
if str(robyn_libraries_path) not in sys.path:
    sys.path.append(str(robyn_libraries_path))
import Robyn_paper_2_defs

pandas.set_option('display.max_columns', 200)
pandas.set_option('display.width', 200)

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)



In [ ]:
# Core paths and shared settings
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
scenario_name = 'maximum'
results_path = base_path / 'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario'
intersections_path = base_path / 'dphil_paper_3/results/01_hazard_infrastructure_network_intersections/coastal_flood_network_intersections'
direct_damages_path = results_path / 'direct_damages'
damage_estimates_path = results_path / 'damage_estimates'
weighted_method_label = 'signed_area_distance_5000m_nn_fallback'
weighted_attribution_dir = damage_estimates_path / 'mangrove_attribution_area_distance_all_sectors_signed'
network_csv = base_path / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'

jamaica_metric_grid_crs = 'EPSG:3448'
jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'
mangrove_buffer_m = 5000
map_display_quantile = 0.995
residual_display_tolerance_usd = 1e-6
weighted_method_description = (
    f'weighted area-distance within {mangrove_buffer_m} m + nearest-neighbour fallback beyond the buffer'
)

if not direct_damages_path.exists():
    raise FileNotFoundError(f'Missing folder: {direct_damages_path}')
if not network_csv.exists():
    raise FileNotFoundError(f'Missing file: {network_csv}')
if not intersections_path.exists():
    raise FileNotFoundError(f'Missing folder: {intersections_path}')
if not jamaica_boundary_path.exists():
    raise FileNotFoundError(f'Missing Jamaica boundary file: {jamaica_boundary_path}')
if not weighted_attribution_dir.exists():
    raise FileNotFoundError(
        f'Missing weighted attribution folder: {weighted_attribution_dir}. Run notebook 22 or 25 first.'
    )

jamaica_boundary = geopandas.read_file(jamaica_boundary_path).to_crs(jamaica_metric_grid_crs)

print(f'scenario_name: {scenario_name}')
print(f'direct_damages_path: {direct_damages_path}')
print(f'network_csv: {network_csv}')
print(f'intersections_path: {intersections_path}')
print(f'weighted_attribution_dir: {weighted_attribution_dir}')


In [ ]:
# Read network metadata and build expected file mapping (asset/layer/id column)
network_details = pandas.read_csv(network_csv)
required_cols = ['asset_gpkg', 'asset_layer', 'asset_description', 'asset_id_column', 'sector']
missing_cols = [c for c in required_cols if c not in network_details.columns]
if missing_cols:
    raise KeyError(f'Missing required columns in network csv: {missing_cols}')

network_details = network_details[required_cols].drop_duplicates().copy()
network_details['folder_name'] = network_details['asset_gpkg'] + '_' + network_details['asset_layer']
network_details['expected_parquet'] = network_details['folder_name'].apply(
    lambda folder: direct_damages_path / folder / f'{folder}_direct_damages_parameter_set_0.parquet'
)
network_details['exists'] = network_details['expected_parquet'].apply(lambda p: p.exists())

display(network_details[['sector', 'asset_description', 'asset_gpkg', 'asset_layer', 'asset_id_column', 'exists']].sort_values(['sector', 'asset_gpkg', 'asset_layer']))

missing_files = network_details.loc[~network_details['exists'], ['asset_gpkg', 'asset_layer', 'expected_parquet']]
if len(missing_files) > 0:
    print('Missing expected files:')
    display(missing_files)
else:
    print('All expected direct-damage files are present.')


In [ ]:
# Load asset-level EAD output from step 05
asset_out = damage_estimates_path / 'coastal_ead_asset_level_usd.csv'
if not asset_out.exists():
    raise FileNotFoundError(f'Missing asset-level EAD CSV: {asset_out}. Run step 05 calculations notebook first.')
asset_ead = pandas.read_csv(asset_out)
print(f'Loaded asset EAD rows: {len(asset_ead):,} from {asset_out}')
display(asset_ead.head(10))



In [ ]:
# Build geospatial layers for mapping sector avoided EAD using coastal split geometries (USD)
if 'asset_ead' not in globals():
    raise ValueError('Run EAD computation cells first so asset_ead exists in memory.')

network_map_details = network_details[[
    'sector', 'asset_description', 'asset_gpkg', 'asset_layer', 'asset_id_column'
]].drop_duplicates().copy()

map_layers = []
missing_split_files = []

for row in network_map_details.itertuples(index=False):
    split_file = intersections_path / f"{row.asset_gpkg}_splits__coastal_flood_rasters_for_intersections__{row.asset_layer}.geoparquet"
    if not split_file.exists():
        missing_split_files.append(str(split_file))
        continue

    split_geom = geopandas.read_parquet(split_file)
    if split_geom.crs is not None:
        split_geom = split_geom.to_crs(jamaica_metric_grid_crs)
    if row.asset_id_column not in split_geom.columns:
        print(f"Skipping {row.asset_gpkg}_{row.asset_layer}: id column '{row.asset_id_column}' not in split file")
        continue

    split_geom = split_geom[[row.asset_id_column, 'geometry']].copy()
    split_geom = geopandas.GeoDataFrame(split_geom, geometry='geometry', crs=split_geom.crs)

    ead_subset = asset_ead.loc[
        (asset_ead['Asset'] == row.asset_gpkg) & (asset_ead['Layer'] == row.asset_layer),
        ['Asset_ID', 'Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD']
    ].copy()

    if ead_subset.empty:
        continue

    split_geom['_join_id'] = split_geom[row.asset_id_column].astype(str)
    ead_subset['_join_id'] = ead_subset['Asset_ID'].astype(str)

    merged = split_geom.merge(
        ead_subset[['_join_id', 'Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD']],
        on='_join_id',
        how='left'
    )

    merged['Sector'] = row.sector
    merged['Subsector'] = row.asset_description
    merged['Asset'] = row.asset_gpkg
    merged['Layer'] = row.asset_layer
    merged['Asset_ID'] = merged[row.asset_id_column]
    merged['Avoided_EAD_USD'] = merged['Avoided_EAD_USD'].fillna(0.0)

    map_layers.append(merged[[
        'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
        'Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'geometry'
    ]])

if not map_layers:
    raise ValueError('No map layers could be built from split files.')

sector_avoided_ead_map_layers = geopandas.GeoDataFrame(
    pandas.concat(map_layers, ignore_index=True),
    geometry='geometry',
    crs=jamaica_metric_grid_crs
)

print(f"Map features loaded: {len(sector_avoided_ead_map_layers):,}")
print('Features by sector:')
display(sector_avoided_ead_map_layers.groupby('Sector', as_index=False).size())

if missing_split_files:
    print('Missing split files (skipped):')
    for p in sorted(set(missing_split_files)):
        print('-', p)



In [ ]:
# Load weighted attribution outputs and build report tables
if 'sector_avoided_ead_map_layers' not in globals():
    raise ValueError('Run the map-layer build cell first so sector_avoided_ead_map_layers exists.')

asset_key_columns = ['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID']
weighted_output_paths = {
    'asset_status': weighted_attribution_dir / f'asset_attribution_status_{weighted_method_label}.csv',
    'asset_patch_summary': weighted_attribution_dir / f'asset_patch_count_and_ids_all_sectors_{weighted_method_label}.csv',
    'sector_breakdown': weighted_attribution_dir / f'attribution_breakdown_by_sector_{weighted_method_label}.csv',
    'subsector_breakdown': weighted_attribution_dir / f'attribution_breakdown_by_subsector_{weighted_method_label}.csv',
    'sector_signed_profile': weighted_attribution_dir / f'sector_attribution_net_sign_profile_{weighted_method_label}.csv',
    'subsector_signed_profile': weighted_attribution_dir / f'subsector_attribution_net_sign_profile_{weighted_method_label}.csv',
    'mangrove_signed_profile': weighted_attribution_dir / f'mangrove_attribution_net_sign_profile_all_sectors_{weighted_method_label}.csv',
    'mangrove_sector_summary': weighted_attribution_dir / f'mangrove_attribution_by_sector_all_sectors_{weighted_method_label}.csv',
    'mangrove_patch_net_sign_summary': weighted_attribution_dir / f'mangrove_patch_net_sign_summary_all_sectors_{weighted_method_label}.csv',
    'positive_ranking': weighted_attribution_dir / f'mangrove_attribution_positive_ranking_all_sectors_{weighted_method_label}.csv',
    'negative_ranking': weighted_attribution_dir / f'mangrove_attribution_negative_ranking_all_sectors_{weighted_method_label}.csv',
    'absolute_ranking': weighted_attribution_dir / f'mangrove_attribution_absolute_ranking_all_sectors_{weighted_method_label}.csv',
    'run_summary': weighted_attribution_dir / f'run_summary_all_sectors_{weighted_method_label}.csv',
    'mangrove_gpkg': weighted_attribution_dir / f'mangrove_attribution_total_all_sectors_{weighted_method_label}.gpkg',
}
missing_output_paths = [
    f'{path_key}: {path_value}'
    for path_key, path_value in weighted_output_paths.items()
    if not path_value.exists()
]
if missing_output_paths:
    raise FileNotFoundError(
        'Missing weighted attribution outputs. Run notebook 22 or 25 first:\n' + '\n'.join(missing_output_paths)
    )

asset_status_all = pandas.read_csv(weighted_output_paths['asset_status'])
asset_patch_summary = pandas.read_csv(weighted_output_paths['asset_patch_summary'])
sector_attribution_breakdown = pandas.read_csv(weighted_output_paths['sector_breakdown'])
subsector_attribution_breakdown = pandas.read_csv(weighted_output_paths['subsector_breakdown'])
sector_signed_profile = pandas.read_csv(weighted_output_paths['sector_signed_profile'])
subsector_signed_profile = pandas.read_csv(weighted_output_paths['subsector_signed_profile'])
mangrove_total_summary = pandas.read_csv(weighted_output_paths['mangrove_signed_profile'])
mangrove_sector_summary = pandas.read_csv(weighted_output_paths['mangrove_sector_summary'])
mangrove_patch_net_sign_summary = pandas.read_csv(weighted_output_paths['mangrove_patch_net_sign_summary'])
positive_ranking = pandas.read_csv(weighted_output_paths['positive_ranking'])
negative_ranking = pandas.read_csv(weighted_output_paths['negative_ranking'])
absolute_ranking = pandas.read_csv(weighted_output_paths['absolute_ranking'])
run_summary = pandas.read_csv(weighted_output_paths['run_summary'])
mangrove_attribution_map = geopandas.read_file(weighted_output_paths['mangrove_gpkg']).to_crs(jamaica_metric_grid_crs)

for table_with_asset_id in [asset_status_all, asset_patch_summary]:
    table_with_asset_id['Asset_ID'] = table_with_asset_id['Asset_ID'].astype(str)

for table_with_residuals in [asset_status_all, asset_patch_summary, sector_attribution_breakdown, subsector_attribution_breakdown]:
    table_with_residuals['Unattributed_EAD_USD'] = numpy.where(
        table_with_residuals['Unattributed_EAD_USD'].abs() < residual_display_tolerance_usd,
        0.0,
        table_with_residuals['Unattributed_EAD_USD'],
    )

asset_geometries = sector_avoided_ead_map_layers[
    asset_key_columns + ['Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'geometry']
].copy()
asset_geometries['Asset_ID'] = asset_geometries['Asset_ID'].astype(str)
asset_geometries = asset_geometries.dissolve(
    by=asset_key_columns,
    as_index=False,
    aggfunc={
        'Avoided_EAD_USD': 'first',
        'EAD_With_Mangroves_USD': 'first',
        'EAD_Without_Mangroves_USD': 'first',
    },
)
asset_geometries = geopandas.GeoDataFrame(asset_geometries, geometry='geometry', crs=jamaica_metric_grid_crs)

asset_level_status = asset_status_all.merge(
    asset_geometries[asset_key_columns + ['geometry']],
    on=asset_key_columns,
    how='left',
)
asset_level_status = geopandas.GeoDataFrame(asset_level_status, geometry='geometry', crs=jamaica_metric_grid_crs)
missing_geometry_count = int(asset_level_status['geometry'].isna().sum())
if missing_geometry_count > 0:
    print(f'Warning: missing geometries for {missing_geometry_count:,} attributed assets.')

out_dir = weighted_attribution_dir

print(f'Loaded weighted attribution outputs from: {weighted_attribution_dir}')
print(f'Attribution method: {weighted_method_description}')
print('Run summary:')
display(run_summary)

print('Top 10 mangroves by positive attributed avoided EAD (USD):')
display(positive_ranking.head(10))
print('Top 10 mangroves by most negative attributed avoided EAD (USD):')
display(negative_ranking.head(10))
print('Top 10 assets by mangrove patch count:')
display(
    asset_patch_summary[[
        'Sector',
        'Subsector',
        'Asset_ID',
        'Attributed_EAD_USD',
        'Mangrove_Patch_Count',
        'Used_Nearest_Fallback',
        'Mangrove_ID_List',
    ]].head(10)
)


In [ ]:
# Breakdown of weighted attributed avoided EAD by sector and subsector (USD)
if 'sector_attribution_breakdown' not in globals():
    raise ValueError('Run the weighted attribution output cell first so breakdown tables exist.')

print('Sector attribution breakdown (weighted area-distance + nearest-neighbour fallback):')
display(sector_attribution_breakdown)

print('Subsector attribution breakdown (weighted area-distance + nearest-neighbour fallback):')
display(subsector_attribution_breakdown)

print('Sector signed attribution profile (positive, negative, and net):')
display(sector_signed_profile)

print('Subsector signed attribution profile (positive, negative, and net):')
display(subsector_signed_profile)

print('Mangrove patch sign summary by patch count:')
display(mangrove_patch_net_sign_summary)

print(f"Source: {weighted_output_paths['sector_breakdown']}")
print(f"Source: {weighted_output_paths['subsector_breakdown']}")
print(f"Source: {weighted_output_paths['sector_signed_profile']}")
print(f"Source: {weighted_output_paths['subsector_signed_profile']}")
print(f"Source: {weighted_output_paths['mangrove_patch_net_sign_summary']}")


In [ ]:
# Map positive avoided-EAD assets attributed via nearest-neighbour fallback beyond the buffer
if 'asset_level_status' not in globals():
    raise ValueError('Run the weighted attribution output cell first so asset_level_status exists.')

fallback_assets_map = asset_level_status.loc[
    (asset_level_status['Used_Nearest_Fallback'] == 1)
    & (asset_level_status['Avoided_EAD_USD'] > 0)
    & (asset_level_status['geometry'].notna())
].copy()

if fallback_assets_map.empty:
    print('No positive avoided-EAD assets required nearest-neighbour fallback for this scenario.')
else:
    fallback_assets_map['geometry'] = fallback_assets_map.geometry.representative_point()
    value_column = 'Avoided_EAD_USD'

    values = fallback_assets_map[value_column].fillna(0.0)
    display_cap = float(values.quantile(map_display_quantile))
    if display_cap <= 0:
        display_cap = float(values.max()) if float(values.max()) > 0 else 1.0

    fallback_assets_map['_plot_val'] = values.clip(upper=display_cap)

    green_colormap = LinearSegmentedColormap.from_list(
        'white_to_dark_green',
        ['#eef8ef', '#b5dfb8', '#5fbf6a', '#238b45', '#005a32'],
        N=256,
    )
    value_norm = plt.Normalize(vmin=0.0, vmax=display_cap)
    scaled_marker_sizes = 20 + 16 * numpy.log10(fallback_assets_map['_plot_val'].clip(lower=1e-9) + 1)

    figure, axis = plt.subplots(figsize=(10.8, 9.4))
    axis.set_facecolor('#ffffff')
    jamaica_boundary.boundary.plot(ax=axis, color='#9a9a9a', linewidth=0.5, zorder=1)

    fallback_assets_map.plot(
        ax=axis,
        column='_plot_val',
        cmap=green_colormap,
        norm=value_norm,
        markersize=scaled_marker_sizes,
        alpha=0.95,
        zorder=3,
    )

    scalar_mappable = ScalarMappable(norm=value_norm, cmap=green_colormap)
    scalar_mappable.set_array([])
    colorbar = figure.colorbar(
        scalar_mappable,
        ax=axis,
        orientation='horizontal',
        fraction=0.045,
        pad=0.02,
    )
    colorbar.set_label('Avoided EAD attributed via nearest-neighbour fallback (USD)')
    colorbar.ax.xaxis.set_major_formatter(FuncFormatter(lambda tick_value, tick_position: f'{tick_value:,.0f}'))

    Robyn_paper_2_defs.draw_scale_bar(axis, location=(0.88, 0.78), length_km=20, linewidth=0.6, label_offset=0.02, km_offset=0.01)
    Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)

    axis.set_title(
        f'Positive avoided-EAD assets using nearest-neighbour fallback ({scenario_name} scenario)',
        fontsize=12,
    )
    axis.set_axis_off()
    plt.tight_layout()

    fallback_map_file = out_dir / f'fallback_positive_assets_map_{weighted_method_label}_q{int(map_display_quantile * 1000)}.png'
    figure.savefig(fallback_map_file, dpi=300, bbox_inches='tight')
    print(f'Saved: {fallback_map_file}')
    print(f'Fallback positive assets mapped: {len(fallback_assets_map):,}')
    print(f'Fallback positive avoided EAD total (USD): {float(fallback_assets_map[value_column].sum()):,.2f}')
    plt.show()


In [ ]:
# Map mangrove-level net attributed avoided EAD (USD) using the weighted method
if 'mangrove_attribution_map' not in globals():
    raise ValueError('Run the weighted attribution output cell first so mangrove_attribution_map exists.')

value_column = 'Net_Avoided_EAD_USD_attributed'
if value_column not in mangrove_attribution_map.columns:
    raise KeyError(f"Column '{value_column}' not found in mangrove attribution map data.")

mangrove_plot = mangrove_attribution_map.copy()
values = mangrove_plot[value_column].fillna(0.0)
absolute_values = values.abs()
display_cap = float(absolute_values.quantile(map_display_quantile))
if display_cap <= 0:
    display_cap = float(absolute_values.max()) if float(absolute_values.max()) > 0 else 1.0

mangrove_plot['_plot_value'] = values.clip(lower=-display_cap, upper=display_cap)

red_white_green_colormap = LinearSegmentedColormap.from_list(
    'red_white_green',
    ['#c81e1e', '#ffffff', '#0b8f3f'],
    N=256,
)
value_norm = TwoSlopeNorm(vmin=-display_cap, vcenter=0.0, vmax=display_cap)

figure, axis = plt.subplots(figsize=(10.8, 9.4))
axis.set_facecolor('#ffffff')
jamaica_boundary.boundary.plot(ax=axis, color='#9a9a9a', linewidth=0.5, zorder=1)

mangrove_plot.plot(
    ax=axis,
    column='_plot_value',
    cmap=red_white_green_colormap,
    norm=value_norm,
    edgecolor='#6f6f6f',
    linewidth=0.35,
    alpha=0.98,
    zorder=2,
)

scalar_mappable = ScalarMappable(norm=value_norm, cmap=red_white_green_colormap)
scalar_mappable.set_array([])
colorbar = figure.colorbar(
    scalar_mappable,
    ax=axis,
    orientation='horizontal',
    fraction=0.045,
    pad=0.02,
)
colorbar.set_label('Net attributed avoided EAD (USD) | red = negative, green = positive')
colorbar.ax.xaxis.set_major_formatter(FuncFormatter(lambda tick_value, tick_position: f'{tick_value:,.0f}'))

Robyn_paper_2_defs.draw_scale_bar(axis, location=(0.88, 0.78), length_km=20, linewidth=0.6, label_offset=0.02, km_offset=0.01)
Robyn_paper_2_defs.draw_north_arrow(axis, location=(0.88, 0.86), size=0.05, fontsize=8, label_offset=0.02)

axis.set_title(
    f'Mangrove net attributed avoided EAD ({scenario_name} scenario, weighted + nearest fallback)',
    fontsize=12,
)
axis.set_axis_off()
plt.tight_layout()

mangrove_map_file = out_dir / f'mangrove_attribution_map_step07_{weighted_method_label}_q{int(map_display_quantile * 1000)}.png'
figure.savefig(mangrove_map_file, dpi=300, bbox_inches='tight')
print(f'Saved: {mangrove_map_file}')
plt.show()
